# Project 61 — Local Prompt Evaluation Lab
## Systematic Prompt Variant Testing & Scoring

**Stack:** LangChain · Ollama · pandas · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama pandas

## Step 1 — Define Prompt Variants

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import pandas as pd, time

llm = ChatOllama(model="qwen3:8b", temperature=0.3)

# Define multiple prompt variants for the same task
prompt_variants = {
    "zero_shot": "Answer the question: {question}",
    "role_play": "You are an expert scientist. Answer clearly: {question}",
    "chain_of_thought": "Think step by step, then answer: {question}",
    "structured": """Answer the question following this format:
ANSWER: <direct answer>
CONFIDENCE: <high/medium/low>
REASONING: <brief reasoning>

Question: {question}""",
}

test_questions = [
    "Why is the sky blue?",
    "What causes inflation?",
    "How does a neural network learn?",
]
print(f"Testing {len(prompt_variants)} prompt variants × {len(test_questions)} questions")

## Step 2 — Run All Variants

In [ ]:
results = []
for variant_name, template in prompt_variants.items():
    prompt = ChatPromptTemplate.from_template(template)
    chain = prompt | llm | StrOutputParser()
    for q in test_questions:
        start = time.time()
        answer = chain.invoke({"question": q})
        elapsed = time.time() - start
        results.append({
            "variant": variant_name,
            "question": q[:40],
            "answer_len": len(answer),
            "latency_s": round(elapsed, 2),
            "answer_preview": answer[:120].replace("\n", " "),
        })
        print(f"  {variant_name} | {q[:30]}... | {elapsed:.1f}s | {len(answer)} chars")

df = pd.DataFrame(results)
print(f"\nCollected {len(df)} results")

## Step 3 — Score Answers with LLM-as-Judge

In [ ]:
judge_prompt = ChatPromptTemplate.from_messages([
    ("system", """Rate the answer quality from 1-5 on:
- Accuracy (is it correct?)
- Clarity (is it easy to understand?)
- Completeness (does it fully address the question?)

Return ONLY a JSON: {{"accuracy": N, "clarity": N, "completeness": N}}"""),
    ("human", "Question: {question}\nAnswer: {answer}")
])
judge_chain = judge_prompt | llm | StrOutputParser()

import json
scores = []
for _, row in df.iterrows():
    try:
        raw = judge_chain.invoke({"question": row["question"], "answer": row["answer_preview"]})
        # Extract JSON from response
        start = raw.find("{")
        end = raw.rfind("}") + 1
        if start >= 0 and end > start:
            score = json.loads(raw[start:end])
        else:
            score = {"accuracy": 3, "clarity": 3, "completeness": 3}
    except Exception:
        score = {"accuracy": 3, "clarity": 3, "completeness": 3}
    score["variant"] = row["variant"]
    score["question"] = row["question"]
    scores.append(score)

scores_df = pd.DataFrame(scores)
scores_df["total"] = scores_df["accuracy"] + scores_df["clarity"] + scores_df["completeness"]

## Step 4 — Leaderboard

In [ ]:
leaderboard = scores_df.groupby("variant")[["accuracy","clarity","completeness","total"]].mean().round(2)
leaderboard = leaderboard.sort_values("total", ascending=False)
print("PROMPT VARIANT LEADERBOARD")
print("="*60)
print(leaderboard.to_string())

print("\nBest variant:", leaderboard.index[0])

## What You Learned
- **Systematic prompt evaluation** with multiple variants
- **LLM-as-judge** scoring for quality metrics
- **Quantitative comparison** of prompt strategies